# Notebook Databricks — Ingestão Bronze

**Objetivo:** ler os arquivos JSON brutos gravados no Volume (`lakehouse_catalog.bronze.landing_volume`) pela etapa de scraping e gravá-los, sem transformação, na tabela `lakehouse_catalog.bronze.livros_raw`, com metadados de auditoria.

Pré-requisitos:
- Executar `sql/ddl/00_setup_unity_catalog.sql` e `sql/ddl/01_create_bronze_tables.sql`.
- Executar o notebook `01_scrape_to_volume.ipynb`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

from common.spark_utils import get_spark_session, volume_path
from common.bronze_utils import read_raw_json, add_audit_columns, write_bronze_table

spark = get_spark_session()

In [ ]:
raw_path = volume_path("livros")

df_raw = read_raw_json(spark, raw_path)
df_raw.printSchema()
df_raw.show(5, truncate=False)

In [ ]:
df_bronze = add_audit_columns(df_raw, raw_path)
write_bronze_table(df_bronze, "livros_raw", mode="append")

print(f"Registros gravados em lakehouse_catalog.bronze.livros_raw: {df_bronze.count()}")

In [ ]:
# Validação rápida
spark.sql("SELECT categoria, count(*) as qtd FROM lakehouse_catalog.bronze.livros_raw GROUP BY categoria").show()